# Chunking CLI OCR option check

합성 CSV, traceability JSON, image-only PDF를 임시 폴더에 만들고 기본 실행과 `include_ocr=True` 실행을 비교합니다. 출력에는 건수와 생성 파일명만 표시합니다.

In [ ]:
import csv
import json
import tempfile
from pathlib import Path

import pymupdf
from payment_card.file_pipeline.chunking.cli import run

WORK = Path(tempfile.mkdtemp(prefix="chunking-ocr-check-"))
ISSUER = "hana"
DATA_ROOT = WORK / "data"
INDEX_DIR = WORK / "index"
TRACE_DIR = WORK / "trace"
OUT_DEFAULT = WORK / "out-default"
OUT_OCR = WORK / "out-ocr"
(DATA_ROOT / "docs").mkdir(parents=True)
(INDEX_DIR / ISSUER).mkdir(parents=True)
(TRACE_DIR / ISSUER).mkdir(parents=True)
rows = [
    {"doc_id": "plain", "sha256": "plain-hash", "file_path": "docs/plain.pdf", "canonical": "Y", "doc_role": "product_guide", "card_type": "credit", "ocr_needed": "N"},
    {"doc_id": "ocr", "sha256": "ocr-hash", "file_path": "docs/ocr.pdf", "canonical": "Y", "doc_role": "product_guide", "card_type": "credit", "ocr_needed": "Y"},
]
with (INDEX_DIR / ISSUER / "documents.csv").open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=rows[0].keys())
    writer.writeheader(); writer.writerows(rows)
with (INDEX_DIR / ISSUER / "product_docs.csv").open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["doc_id", "product_name", "product_ids"])
    writer.writeheader()
    writer.writerow({"doc_id": "ocr", "product_name": "Synthetic", "product_ids": "{}"})
for row in rows:
    pdf = pymupdf.open()
    page = pdf.new_page(width=100, height=100)
    pixmap = pymupdf.Pixmap(pymupdf.csRGB, pymupdf.IRect(0, 0, 4, 4), 0)
    pixmap.clear_with(255)
    page.insert_image(page.rect, stream=pixmap.tobytes("png"))
    pdf.save(DATA_ROOT / row["file_path"]); pdf.close()
    trace = {"document": {"sha256": row["sha256"]}, "pages": [{"page_no": 1, "text": f"synthetic {row['doc_id']} page\n"}]}
    (TRACE_DIR / ISSUER / f"{row['doc_id']}.json").write_text(json.dumps(trace), encoding="utf-8")
print("synthetic inputs ready", len(rows), "documents")

## 기본 실행과 OCR 포함 실행 결과

In [ ]:
default = run(ISSUER, DATA_ROOT, INDEX_DIR, TRACE_DIR, OUT_DEFAULT, False)
with_ocr = run(ISSUER, DATA_ROOT, INDEX_DIR, TRACE_DIR, OUT_OCR, False, include_ocr=True)
print({"default_done": len(default["done"]), "default_skipped_ocr": len(default["skipped"])})
print({"with_ocr_done": len(with_ocr["done"]), "with_ocr_skipped": len(with_ocr["skipped"]),
       "included_ocr": with_ocr["included_ocr"],
       "ocr_chunk_file_exists": (OUT_OCR / ISSUER / "ocr.chunks.json").is_file()})